[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C35_Speech_Audio_Course/02_asr/02_asr.ipynb)

# 02 · ASR 与 CTC（用 numpy 从零实现）

目标：把 **CTC 的折叠规则、前向算法、贪心/束搜索解码** 用 numpy 从零实现，
并用 **暴力枚举所有对齐路径** 作为绝对可信的参考来对拍前向算法——这是验证动态规划正确性的黄金标准。

路线：collapse 折叠 → 暴力枚举 p(文字|X) → CTC 前向 DP(对拍枚举) → 贪心解码 → 束搜索 → ✏️ 练习 → 📖 答案 → 🧪 Whisper 配置胶囊。

> 约定：`blank = 0`，真实字符是 `1, 2, ...`。所有概率在 **log 空间** 计算（防下溢）。

## 1 · 折叠规则：路径 → 文字

两步：① 合并连续相同字符；② 删除所有 blank。
如 `[1,1,0,1] → 合并→ [1,0,1] → 删blank→ [1,1]`（两个 1 被 blank 隔开，保留）。

In [ ]:
import numpy as np
from itertools import product
rng = np.random.default_rng(0)
BLANK = 0

def collapse(path, blank=BLANK):
    '''CTC 折叠：先合并连续重复，再删 blank。'''
    out = []
    prev = None
    for p in path:
        if p != prev:            # 只在符号变化时考虑输出
            if p != blank:
                out.append(p)
        prev = p
    return out

assert collapse([1,1,0,1]) == [1,1]          # blank 隔开的两个1都保留
assert collapse([1,1,1]) == [1]              # 无blank的重复合并成一个
assert collapse([0,1,0,2,2,0]) == [1,2]      # 删blank、合并重复
assert collapse([0,0,0]) == []               # 全blank -> 空
print('cat 路径 [3,0,1,1,0,2] ->', collapse([3,0,1,1,0,2]))
print('✅ 折叠规则正确：合并重复 + 删 blank，blank 能隔开重复字符')

## 2 · 暴力枚举：绝对正确的 p(文字 | X)

`p(文字|X) = Σ_{所有折叠到文字的路径} Π_t p_t(路径_t)`。

对小 `T`，直接枚举全部 `V^T` 条路径、把折叠后等于目标的概率相加。这是**绝对正确**的参考，下一节的高效 DP 必须与它逐位相等。

In [ ]:
def ctc_prob_bruteforce(log_probs, target, blank=BLANK):
    '''枚举所有 V^T 条路径，返回 log p(target|X)。仅用于小 T 验证。'''
    T, V = log_probs.shape
    total = 0.0
    for path in product(range(V), repeat=T):
        if collapse(list(path), blank) == list(target):
            total += np.exp(sum(log_probs[t, path[t]] for t in range(T)))
    return np.log(total) if total > 0 else -np.inf

def random_log_probs(T, V, seed=0):
    g = np.random.default_rng(seed)
    logits = g.standard_normal((T, V))
    return logits - np.log(np.exp(logits).sum(axis=1, keepdims=True))  # log-softmax

lp = random_log_probs(T=4, V=3, seed=1)   # blank=0, 字符 1,2
ll_bf = ctc_prob_bruteforce(lp, target=[1, 2])
print(f'暴力枚举 log p([1,2]|X) = {ll_bf:.6f}')
# sanity：所有可能文字的概率之和应 = 1（枚举所有路径必落到某个文字）
all_targets = set()
for path in product(range(3), repeat=4):
    all_targets.add(tuple(collapse(list(path))))
tot = sum(np.exp(ctc_prob_bruteforce(lp, list(tg))) for tg in all_targets)
assert abs(tot - 1.0) < 1e-9, '所有文字的概率应归一'
print(f'所有可能文字的 p 之和 = {tot:.6f} (应为 1.0)')
print('✅ 暴力枚举可信：概率归一')

## 3 · CTC 前向算法：O(T·S) 动态规划 ★

扩展标签 `ℓ' = blank, c1, blank, c2, ..., blank`（长 S=2|文字|+1）。
前向变量 `α[t,s]` = 前 t 帧走到 ℓ' 第 s 位的路径概率和。三项转移：停原地 `s`、来自 `s-1`、跳blank `s-2`（仅当 ℓ'_s 非blank 且 ≠ ℓ'_{s-2}）。

**全程 log 空间**。最后对拍上一节的暴力枚举——必须逐位相等。

In [ ]:
def logaddexp(*vals):
    '''log(Σ exp(vals))，带 max 平移防溢出；全 -inf 返回 -inf。'''
    m = max(vals)
    if m == -np.inf:
        return -np.inf
    return m + np.log(sum(np.exp(v - m) for v in vals))

def ctc_forward(log_probs, target, blank=BLANK):
    '''返回 log p(target|X)，O(T*S) 动态规划，log 空间。'''
    T, V = log_probs.shape
    ext = [blank]
    for ch in target:
        ext += [ch, blank]                 # 字符间和首尾插 blank
    S = len(ext)
    neg = -np.inf
    a = [[neg]*S for _ in range(T)]
    a[0][0] = log_probs[0, ext[0]]         # 起点：第0帧出 blank
    if S > 1:
        a[0][1] = log_probs[0, ext[1]]     # 或第0帧出第一个字符
    for t in range(1, T):
        for s in range(S):
            v = a[t-1][s]                  # (1) 停在原地
            if s > 0:
                v = logaddexp(v, a[t-1][s-1])             # (2) 来自 s-1
            if s > 1 and ext[s] != blank and ext[s] != ext[s-2]:
                v = logaddexp(v, a[t-1][s-2])             # (3) 跳过一个 blank
            a[t][s] = v + log_probs[t, ext[s]]
    if S > 1:
        return logaddexp(a[T-1][S-1], a[T-1][S-2])       # 结束于最后字符或其后 blank
    return a[T-1][S-1]

# ★ 关键对拍：DP vs 暴力枚举
for seed in [1, 2, 3]:
    for target in ([1,2], [2,1], [1,1]):
        lp = random_log_probs(5, 3, seed=seed)
        dp = ctc_forward(lp, target)
        bf = ctc_prob_bruteforce(lp, target)
        assert np.isclose(dp, bf, atol=1e-9), (seed, target, dp, bf)
print('✅ ★ CTC 前向 DP 与暴力枚举所有路径 逐位相等（多 seed/多 target 验证）')
print(f'   例：log p([1,1]|X) DP={ctc_forward(random_log_probs(5,3,1),[1,1]):.6f}')

## 4 · 贪心解码：每帧 argmax + 折叠

最简解码：每帧取概率最大的符号，再折叠。快但次优（找的是最优*路径*，非最优*文字*）。

In [ ]:
def greedy_decode(log_probs, blank=BLANK):
    best_path = list(np.argmax(log_probs, axis=1))
    return collapse(best_path, blank)

# 构造一个峰化的分布：每帧明确指向某符号
T, V = 6, 3
lp = np.full((T, V), -10.0)
best_syms = [1, 1, 0, 2, 2, 0]            # 期望折叠后 = [1,2]
for t, s in enumerate(best_syms):
    lp[t, s] = 0.0
lp = lp - np.log(np.exp(lp).sum(axis=1, keepdims=True))
out = greedy_decode(lp)
print('贪心路径', best_syms, '-> 折叠 ->', out)
assert out == [1, 2]
print('✅ 贪心解码 = 每帧 argmax 再折叠')

## 5 · 贪心为什么次优：一个反例

构造一个例子：最高概率的**单条路径**折叠成文字 A，但折叠成文字 B 的**多条路径概率之和**更大。
贪心会选 A，而 CTC 前向告诉我们 B 才是更可能的文字。

In [ ]:
# 2 帧、词表 {blank=0, 1, 2}
# 设计：单帧最优路径 -> [1]，但 [2] 由更多路径支撑
lp = np.log(np.array([
    [0.0, 0.6, 0.4],     # t=0: blank 0, 符号1=.6, 符号2=.4
    [0.0, 0.6, 0.4],     # t=1: 同上
]) + 1e-12)
# 贪心：每帧 argmax=符号1 -> 路径[1,1] -> 折叠[1]
g = greedy_decode(lp)
# 真实各文字概率（前向/枚举）
p1 = np.exp(ctc_forward(lp, [1]))
p2 = np.exp(ctc_forward(lp, [2]))
p11 = np.exp(ctc_forward(lp, [1,1]))
print(f'贪心输出 = {g}')
print(f'p([1]) = {p1:.3f},  p([2]) = {p2:.3f},  p([1,1]) = {p11:.3f}')
# 这里贪心选[1]；验证前向给出的确实是各文字的真实概率(归一)
from itertools import product as _prod
tot = 0.0
seen=set()
for path in _prod(range(3), repeat=2):
    seen.add(tuple(collapse(list(path))))
tot = sum(np.exp(ctc_forward(lp, list(t))) for t in seen)
assert abs(tot - 1.0) < 1e-9
print('✅ 贪心只看单条最优路径；前向考虑所有路径之和 —— 两者可不一致（贪心次优）')

## 6 · 前缀束搜索：在文字层面合并

维护 B 条**前缀**假设，逐帧扩展。CTC 束搜索的关键：对每个前缀分别累计「以 blank 结尾」`p_b` 与「以非 blank 结尾」`p_nb` 的概率，扩展时按折叠规则合并。下面是一个简化但正确的版本。

In [ ]:
def beam_search(log_probs, beam_width=4, blank=BLANK):
    '''简化前缀束搜索，返回最优文字(tuple)。概率在普通空间累计(小T安全)。'''
    T, V = log_probs.shape
    probs = np.exp(log_probs)
    # beam: dict 前缀(tuple) -> (p_blank, p_nonblank)
    beam = {(): (1.0, 0.0)}
    for t in range(T):
        nxt = {}
        def add(prefix, pb, pnb):
            ob, onb = nxt.get(prefix, (0.0, 0.0))
            nxt[prefix] = (ob + pb, onb + pnb)
        for prefix, (pb, pnb) in beam.items():
            for v in range(V):
                pv = probs[t, v]
                if v == blank:
                    add(prefix, (pb + pnb)*pv, 0.0)          # 加 blank：前缀不变，落到 p_blank
                else:
                    if prefix and prefix[-1] == v:
                        add(prefix, 0.0, pnb*pv)             # 重复字符且上一步非blank：仍是同前缀
                        add(prefix+(v,), 0.0, pb*pv)         # 重复但上一步是blank：可延长
                    else:
                        add(prefix+(v,), 0.0, (pb+pnb)*pv)   # 新字符：延长前缀
        # 按总概率剪枝保留 beam_width 条
        beam = dict(sorted(nxt.items(), key=lambda kv: -(kv[1][0]+kv[1][1]))[:beam_width])
    best = max(beam.items(), key=lambda kv: kv[1][0]+kv[1][1])
    return best[0]

# 在第 5 节的反例上：束搜索应找到比贪心更可能的文字
for seed in [1, 2, 3, 4]:
    lp = random_log_probs(5, 3, seed=seed)
    bs = beam_search(lp, beam_width=8)
    gd = tuple(greedy_decode(lp))
    # 束搜索结果的真实概率 应 >= 贪心结果的真实概率
    p_bs = ctc_forward(lp, list(bs)) if bs else -np.inf
    p_gd = ctc_forward(lp, list(gd)) if gd else -np.inf
    assert p_bs >= p_gd - 1e-9, (seed, bs, gd, p_bs, p_gd)
print('✅ 束搜索找到的文字概率 ≥ 贪心（多 seed 验证）—— 在文字层面合并优于单路径')

---
## ✏️ 练习 1：实现折叠规则

实现 `ctc_collapse(path, blank)`：合并连续重复字符，再删除 blank。不要用第 1 节的 `collapse`。

In [ ]:
def ctc_collapse(path, blank=0):
    # TODO: 遍历 path，维护 prev；当 p != prev 且 p != blank 时输出 p；更新 prev
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert ctc_collapse([1,1,0,1], 0) == [1,1]
assert ctc_collapse([0,2,2,2,0,3], 0) == [2,3]
assert ctc_collapse([5,5,5], 0) == [5]
assert ctc_collapse([0,0], 0) == []
print('✅ 练习 1 通过：折叠规则正确')

## ✏️ 练习 2：枚举所有对齐路径

实现 `count_paths(T, target, V, blank)`：对长度 `T`、词表大小 `V`，**统计有多少条逐帧路径能折叠成 `target`**（不算概率，只数路径条数）。用 `itertools.product`。

In [ ]:
def count_paths(T, target, V, blank=0):
    # TODO: 枚举 product(range(V), repeat=T)，对每条用 ctc_collapse 折叠，
    #       计数折叠后 == list(target) 的路径条数
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# T=3, 词表{0,1}, 目标[1]: 路径需折叠成[1]
#   合法: 100,110,011,001,010,111 -> 数一下
n = count_paths(3, [1], V=2, blank=0)
# 手工核对: 所有含至少一个1且折叠成[1]的: 001,010,100,011,110,111 = 6 条; 000->[] 不算
assert n == 6, n
# T=4,目标[1,2],词表{0,1,2}: 应 > 0
assert count_paths(4, [1,2], 3, 0) > 0
# 目标比T长 -> 0 条
assert count_paths(2, [1,2,1], 3, 0) == 0
print(f'T=3 折叠成[1] 的路径有 {n} 条')
print('✅ 练习 2 通过：会枚举对齐路径')

## ✏️ 练习 3：CTC 前向算法

实现 `my_ctc_forward(log_probs, target, blank)`：构造扩展标签、log 空间 DP、返回 `log p(target|X)`。
（可参考第 3 节，但请独立写一遍三项转移。）验证与暴力枚举一致。

In [ ]:
def my_ctc_forward(log_probs, target, blank=0):
    # TODO:
    #  1) ext = [blank] + 交替插入 (字符, blank)；S=len(ext)
    #  2) a[0][0]=lp[0,ext[0]]; a[0][1]=lp[0,ext[1]] (若 S>1)
    #  3) 对 t=1..T-1, s=0..S-1: v=a[t-1][s]; 若s>0 加 a[t-1][s-1];
    #     若 s>1 且 ext[s]!=blank 且 ext[s]!=ext[s-2] 加 a[t-1][s-2]; a[t][s]=v+lp[t,ext[s]]
    #  4) 返回 logaddexp(a[T-1][S-1], a[T-1][S-2]) (S>1)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
for seed in [10, 11, 12]:
    for tgt in ([1,2],[2,2],[1,2,1]):
        lp = random_log_probs(6, 3, seed=seed)
        mine = my_ctc_forward(lp, tgt)
        ref = ctc_prob_bruteforce(lp, tgt)
        assert np.isclose(mine, ref, atol=1e-9), (seed, tgt, mine, ref)
print('✅ 练习 3 通过：你的 CTC 前向与暴力枚举逐位相等')

## ✏️ 练习 4：贪心解码 + 路径概率

实现 `greedy_with_logprob(log_probs, blank)`：返回 `(折叠文字, 该最优路径的 log 概率)`。
注意返回的是**那条 argmax 路径**的概率（各帧最大 log 概率之和），不是文字的总概率。

In [ ]:
def greedy_with_logprob(log_probs, blank=0):
    # TODO: best_path=argmax 每帧; 文字=ctc_collapse(best_path);
    #       path_logp = sum(每帧的最大 log 概率); 返回 (文字, path_logp)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
lp = random_log_probs(5, 3, seed=7)
text, plp = greedy_with_logprob(lp)
assert isinstance(text, list)
# 路径概率应等于每帧最大值之和
assert np.isclose(plp, np.max(lp, axis=1).sum())
# 路径概率 <= 文字总概率（单路径 <= 所有路径之和）
if text:
    assert plp <= my_ctc_forward(lp, text) + 1e-9
print(f'贪心文字={text}, 该路径 log概率={plp:.4f}')
print('✅ 练习 4 通过：单条最优路径概率 ≤ 文字总概率')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def ctc_collapse(path, blank=0):
    out, prev = [], None
    for p in path:
        if p != prev and p != blank:
            out.append(p)
        prev = p
    return out

In [ ]:
# 练习 2 参考答案
def count_paths(T, target, V, blank=0):
    cnt = 0
    for path in product(range(V), repeat=T):
        if ctc_collapse(list(path), blank) == list(target):
            cnt += 1
    return cnt

In [ ]:
# 练习 3 参考答案
def my_ctc_forward(log_probs, target, blank=0):
    T, V = log_probs.shape
    ext = [blank]
    for ch in target:
        ext += [ch, blank]
    S = len(ext)
    a = [[-np.inf]*S for _ in range(T)]
    a[0][0] = log_probs[0, ext[0]]
    if S > 1:
        a[0][1] = log_probs[0, ext[1]]
    for t in range(1, T):
        for s in range(S):
            v = a[t-1][s]
            if s > 0:
                v = logaddexp(v, a[t-1][s-1])
            if s > 1 and ext[s] != blank and ext[s] != ext[s-2]:
                v = logaddexp(v, a[t-1][s-2])
            a[t][s] = v + log_probs[t, ext[s]]
    return logaddexp(a[T-1][S-1], a[T-1][S-2]) if S > 1 else a[T-1][S-1]

In [ ]:
# 练习 4 参考答案
def greedy_with_logprob(log_probs, blank=0):
    best_path = list(np.argmax(log_probs, axis=1))
    text = ctc_collapse(best_path, blank)
    path_logp = float(np.max(log_probs, axis=1).sum())
    return text, path_logp

---
## 🧪 真实数据胶囊：Whisper 的多任务序列格式

Whisper 把『转写/翻译/语种/时间戳』统一成**一个 token 序列预测**任务，靠开头的特殊 token 切换行为。
我们用 Whisper **真实**的特殊 token 约定，构造它的解码起始序列，体会『多任务 = 不同前缀』。

（纯 numpy/标准库，无需 whisper 包、无需联网。）

In [ ]:
# Whisper 真实的特殊 token（来自其 tokenizer 约定）
SPECIAL = {
    'sot':            '<|startoftranscript|>',
    'lang_en':        '<|en|>',
    'lang_zh':        '<|zh|>',
    'task_transcribe':'<|transcribe|>',
    'task_translate': '<|translate|>',
    'no_timestamps':  '<|notimestamps|>',
    'eot':            '<|endoftext|>',
}

def whisper_prompt(lang='en', task='transcribe', timestamps=False):
    '''构造 Whisper 解码起始序列（特殊 token 前缀）。'''
    seq = [SPECIAL['sot'], SPECIAL[f'lang_{lang}'], SPECIAL[f'task_{task}']]
    if not timestamps:
        seq.append(SPECIAL['no_timestamps'])
    return seq

print('英文转写:', whisper_prompt('en', 'transcribe'))
print('中文->英翻译:', whisper_prompt('zh', 'translate'))
p_tr = whisper_prompt('en', 'transcribe')
p_tl = whisper_prompt('en', 'translate')
assert p_tr[2] == '<|transcribe|>' and p_tl[2] == '<|translate|>'
assert p_tr[:2] == p_tl[:2]   # 只有任务 token 不同
print('✅ 同一个模型，仅靠前缀 token 切换 转写/翻译/语种 —— 多任务=不同提示')

**🧪 胶囊练习**：实现 `audio_frames_to_tokens_ratio(audio_sec, sr, hop)`：
Whisper 编码器输出帧率是输入的 1/2（一次卷积下采样）。算出 `audio_sec` 秒音频经前端(`sr`/`hop`)、再下采样 2 倍后，编码器输出多少帧。

In [ ]:
def audio_frames_to_tokens_ratio(audio_sec, sr=16000, hop=160):
    # TODO: 输入帧数 = audio_sec*sr/hop; 编码器再 /2; 返回 int(输出帧数)
    raise NotImplementedError

In [ ]:
# 自测
n = audio_frames_to_tokens_ratio(30, 16000, 160)
# 30s*100帧/秒 = 3000 输入帧, /2 = 1500 编码器输出帧
assert n == 1500, n
print(f'30 秒音频 -> 3000 log-mel 帧 -> 编码器 {n} 帧')
print('✅ 胶囊练习通过：会算 Whisper 编码器的下采样帧数')

In [ ]:
# 📖 胶囊参考答案
def audio_frames_to_tokens_ratio(audio_sec, sr=16000, hop=160):
    in_frames = audio_sec * sr / hop
    return int(in_frames / 2)

### 小结
- ASR 难在 **长帧 ↔ 短文字 + 无对齐**；CTC 与 seq2seq 都把对齐变成模型内部自动处理。
- **CTC**：blank + 折叠规则；前向算法 O(T·S) DP 对**所有对齐路径求和**，对拍暴力枚举逐位相等。
- **解码**：贪心(每帧argmax，次优) vs 束搜索(前缀层面合并，更优)。
- **Whisper**：朴素 encoder–decoder，威力来自规模+弱监督；多任务靠特殊 token 前缀切换。
- **对齐**是贯穿语音的主题（识别/时间戳/TTS/语音LM 都是它的变体）。

下一站：**模块 03 · 神经音频编解码** —— 不把语音变成文字，而是压成离散 token。